# 03 · Evaluación y contraste con las dimensiones

**Objetivo específico 3.** Evaluar la correspondencia de los temas con las dimensiones
predefinidas y su relación con la calificación, mediante medidas de acuerdo y análisis
descriptivo, para derivar orientaciones de comunicación digital.

El notebook mide la calidad del modelo, cruza sus tópicos contra las seis dimensiones,
calcula el AMI con su línea base, describe la relación con la calificación, cuantifica
los temas transversales y cierra con las limitaciones que condicionan todo lo anterior.

In [ ]:
# Configuración local. El kernel es el del entorno de uv:
#     uv run python -m ipykernel install --user --name topicos-popayan
import sys
from pathlib import Path

RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(RAIZ))
print(f"raíz del proyecto: {RAIZ}")

In [ ]:
import numpy as np
import pandas as pd
from sklearn.metrics import adjusted_mutual_info_score, adjusted_rand_score

from src.config import DIMENSIONES, DATOS_PROCESADOS, FIGURAS, SEMILLA
from src.topicos import ATIPICO

pd.set_option("display.max_colwidth", 100)

fragmentos = pd.read_csv(DATOS_PROCESADOS / "fragmentos.csv").merge(
    pd.read_csv(DATOS_PROCESADOS / "fragmentos_topicos.csv"),
    on=["fragmento_id", "review_id"],
)
etiquetas = pd.read_csv(DATOS_PROCESADOS / "topicos_etiquetados.csv")
temas = pd.read_csv(DATOS_PROCESADOS / "temas_consolidado.csv")
asignados = fragmentos[fragmentos["topico"] != ATIPICO]
print(f"{len(fragmentos)} fragmentos · {len(asignados)} asignados · {len(etiquetas)} tópicos")

## 1. Calidad del modelo

Se reportan **dos** coherencias a propósito. `c_v` (Röder et al., 2015) es la medida de
referencia y permite comparar con la literatura; `c_npmi` es más conservadora y se le
reprocha menos sesgo optimista. La diversidad (Dieng et al., 2020) es la proporción de
palabras distintas entre las principales de todos los tópicos.

In [ ]:
calidad = pd.read_csv(DATOS_PROCESADOS / "calidad_modelo.csv")
calidad

**Interpretación, sin adornos.** Un `c_v` de 0,484 es moderado: aceptable para texto
corto, lejos del 0,55-0,65 que suele considerarse bueno. **El `c_npmi` cercano a cero es
el dato incómodo**: dice que los términos principales de un tópico no coaparecen en el
mismo fragmento mucho más de lo que cabría por azar.

Hay una causa estructural —el fragmento mediano tiene 10 palabras, así que dos términos
del tópico rara vez caben en el mismo documento, y la medida penaliza la unidad de
análisis elegida— pero **no se puede afirmar que los tópicos sean léxicamente
coherentes**. Lo que sostiene su interpretación es la lectura de fragmentos, no el
indicador.

La diversidad de 0,664 significa que un tercio de los términos principales se repite
entre tópicos, lo que concuerda con la redundancia detectada al evaluar la fusión.

In [ ]:
coherencia_topico = etiquetas.dropna(subset=["c_v"]).sort_values("c_v")
columnas = ["topico_id", "tema", "subtema", "n", "c_v"]
print("Menos coherentes:")
print(coherencia_topico.head(6)[columnas].fillna("").to_string(index=False))
print("\nMás coherentes:")
print(coherencia_topico.tail(6)[columnas].fillna("").to_string(index=False))

Los menos coherentes son los tópicos de plato y de oferta —bebidas, carta y menú, café,
hamburguesas—, y tiene sentido: su vocabulario es una lista de cosas que no coaparecen
(«limonada, vinos, jugo, mango»). Los más coherentes son los grandes tópicos de
dimensión. La coherencia léxica mide lo que puede medir, no la calidad temática.

## 2. Tabla cruzada: tópico por dimensión

Para cada tópico, el porcentaje de sus fragmentos que activa cada dimensión del
diccionario. Un tópico puede activar varias, porque las dimensiones no son excluyentes.

In [ ]:
cruzada = pd.read_csv(DATOS_PROCESADOS / "cruzada_topico_dimension.csv")
cruzada.sort_values("n", ascending=False).head(15)

In [ ]:
from IPython.display import Image, display

display(Image(filename=str(FIGURAS / "03_mapa_topico_dimension.png"), width=850))

### En cuántos tópicos se reparte cada dimensión

In [ ]:
dispersion = []
for dimension in DIMENSIONES:
    sub = asignados[asignados[dimension]]
    reparto = sub["topico"].value_counts()
    dispersion.append({
        "dimension": dimension,
        "fragmentos": len(sub),
        "topicos": len(reparto),
        "% en el mayor": round(100 * reparto.iloc[0] / len(sub), 1),
        "% en los 3 mayores": round(100 * reparto.head(3).sum() / len(sub), 1),
        "topico principal": f"T{reparto.index[0]}",
    })
pd.DataFrame(dispersion).sort_values("% en el mayor", ascending=False)

**Ninguna dimensión se concentra en un solo tópico.** La espera es la única con
correspondencia fuerte —casi dos tercios en un tópico—, y era previsible porque es un
asunto acotado y con vocabulario propio. **Comida es el caso opuesto**: se reparte en 38
tópicos y su tópico principal ni siquiera es de comida, sino el de experiencia global.
Es la dimensión más grande y la menos específica: casi cualquier elogio la menciona.

## 3. AMI entre tópicos y dimensiones

Se calcula sobre los fragmentos con **exactamente una** dimensión activa, que es donde
la comparación tiene sentido: si un fragmento activa tres dimensiones, no hay una
etiqueta única contra la cual contrastar el tópico.

La línea base se estima permutando la etiqueta de dimensión 200 veces, lo que deja
intactos los tamaños de grupo. Sin esa referencia, un AMI no se puede interpretar.

In [ ]:
una = fragmentos[fragmentos["n_dim"] == 1].copy()
una["dimension"] = una[list(DIMENSIONES)].idxmax(axis=1)
con_topico = una[una["topico"] != ATIPICO]

def nulo(topicos, dimensiones, repeticiones=200):
    generador = np.random.default_rng(SEMILLA)
    valores = [adjusted_mutual_info_score(topicos, generador.permutation(dimensiones))
               for _ in range(repeticiones)]
    return float(np.mean(valores)), float(np.std(valores))

filas = []
for etiqueta, base in [("solo fragmentos asignados", con_topico),
                       ("atípicos como categoría", una)]:
    ami = adjusted_mutual_info_score(base["topico"], base["dimension"])
    media, desviacion = nulo(base["topico"].to_numpy(), base["dimension"].to_numpy())
    filas.append({
        "base": etiqueta, "n": len(base), "AMI": round(ami, 4),
        "ARI": round(adjusted_rand_score(base["topico"], base["dimension"]), 4),
        "nulo": round(media, 4), "sd nulo": round(desviacion, 4),
        "exceso": round(ami - media, 4),
    })
pd.DataFrame(filas)

In [ ]:
print(f"cobertura: {len(una)} fragmentos con exactamente una dimensión "
      f"({100 * len(una) / len(fragmentos):.1f} % del corpus)")
print(f"de ellos asignados a un tópico: {len(con_topico)} "
      f"({100 * len(con_topico) / len(una):.1f} %)")

**Hay acuerdo moderado, no equivalencia.** Un AMI de 0,41 sobre un nulo de cero dice que
los tópicos y las dimensiones comparten estructura real, pero que **los tópicos no son
una redescripción de las seis dimensiones**. El ARI, mucho más bajo, lo confirma desde
otro ángulo: la coincidencia por pares es escasa, porque un tópico suele repartirse
entre varias dimensiones y una dimensión entre muchos tópicos.

Es exactamente el resultado que el objetivo 3 buscaba establecer: correspondencia
parcial, con un remanente que el esquema a priori no cubre.

### Cuánto de los atributos emergentes es invisible para el diccionario

In [ ]:
nuevos = etiquetas.loc[etiquetas["tipo"] == "atributo_nuevo", "topico_id"]
sub = asignados[asignados["topico"].isin(nuevos)]
resto = asignados[~asignados["topico"].isin(nuevos)]
print(f"tópicos de atributo emergente: {len(nuevos)} · {len(sub)} fragmentos")
print(f"  sin ninguna dimensión: {100 * (sub['n_dim'] == 0).mean():.1f} %")
print(f"  resto del corpus asignado: {100 * (resto['n_dim'] == 0).mean():.1f} %")

detalle = sub.groupby("topico").agg(
    n=("fragmento_id", "size"),
    pct_sin_dimension=("n_dim", lambda s: round(100 * (s == 0).mean(), 1)),
).join(etiquetas.set_index("topico_id")["tema"])
detalle.sort_values("pct_sin_dimension", ascending=False)

**Medios de pago e infraestructura son invisibles para el diccionario** —el 91 % y el
76 % de sus fragmentos no activan ninguna dimensión—: son atributos nuevos en sentido
estricto. Ocasión de consumo y referente en la ciudad sí rozan el diccionario, porque
traen palabras de ambiente y comida, pero lo que los agrupa —para quién es el sitio, qué
lugar ocupa en la ciudad— no está en el esquema.

## 4. Relación con la calificación

**Descriptiva y solo descriptiva.** Los fragmentos de una misma reseña comparten
calificación y no son independientes, así que cualquier prueba de significancia que
asuma independencia daría errores estándar demasiado pequeños.

In [ ]:
ranking = temas.sort_values("rating", ascending=False)[
    ["tema", "tipo", "fragmentos", "rating", "pct_1_2_estrellas", "polaridad_dominante"]
]
print(f"media del corpus: {fragmentos['rating'].mean():.2f} · "
      f"1-2 estrellas: {100 * (fragmentos['rating'] <= 2).mean():.1f} %")
ranking

In [ ]:
display(Image(filename=str(FIGURAS / "02_temas_por_rating.png"), width=900))

**La espera es el único tema que cae por debajo de tres estrellas** (1,88, con el 75,8 %
de reseñas de una o dos estrellas) y concentra el malestar del corpus. En el extremo
opuesto, intención de volver y satisfacción no son atributos sino veredictos, así que su
rating alto no dice nada sobre qué hace bien un establecimiento.

Entre los atributos, los mejor calificados son **ocasión de consumo (4,58) y referente en
la ciudad (4,55)**, los dos emergentes. Es el insumo más directo para orientar la
comunicación digital: son los asuntos por los que los comensales valoran mejor y que el
esquema tradicional no contempla.

In [ ]:
atipicos = fragmentos[fragmentos["topico"] == ATIPICO]
print(f"atípicos: {len(atipicos)} ({100 * len(atipicos) / len(fragmentos):.1f} %)")
print(f"  rating medio {atipicos['rating'].mean():.2f} · "
      f"1-2 estrellas {100 * (atipicos['rating'] <= 2).mean():.1f} %")
print(f"  corpus: {fragmentos['rating'].mean():.2f} · "
      f"{100 * (fragmentos['rating'] <= 2).mean():.1f} %")
print(f"\n% atípico según dimensiones activas:")
for n_dim, grupo in fragmentos.groupby(fragmentos["n_dim"].clip(upper=2)):
    nombre = {0: "n_dim = 0", 1: "n_dim = 1", 2: "n_dim >= 2"}[n_dim]
    print(f"  {nombre}: {100 * (grupo['topico'] == ATIPICO).mean():>5.1f} %  (n={len(grupo)})")

**Lo que queda sin agrupar es más negativo que el corpus** y está sobrerrepresentado
entre los fragmentos que el diccionario no ve. Las dos cosas apuntan al mismo sitio: la
señal que ni el esquema ni el modelo capturan es desproporcionadamente crítica.

## 5. Los temas transversales

Cuatro asuntos aparecen en el corpus, son de los más negativos, y **no forman tópico con
ningún umbral probado**. Se cuantifican con reglas de recuperación.

> **Advertencia obligatoria.** Estas reglas son de **piso, no una medición**. Cada tema
> se recupera con una expresión regular estrecha sobre el texto; tiene falsos negativos
> (formulaciones que no usan esas palabras) y falsos positivos documentados. Las cifras
> acotan el orden de magnitud. No deben presentarse como prevalencias.

In [ ]:
baja_masa = pd.read_csv(DATOS_PROCESADOS / "temas_baja_masa.csv")
baja_masa[["tema", "fragmentos", "pct_corpus", "resenas", "rating",
           "pct_1_2_estrellas", "pct_atipicos", "topicos_distintos"]]

In [ ]:
ejemplos = pd.read_csv(DATOS_PROCESADOS / "temas_baja_masa_ejemplos.csv")
for tema, grupo in ejemplos.groupby("tema", sort=False):
    print(f"{tema}:")
    for fila in grupo.itertuples(index=False):
        print(f"   ({fila.rating}★) {fila.texto[:115]}")
    print()

**Inocuidad (1,62) y honestidad en el cobro (1,87) están por debajo incluso de la
espera**, que es el tema negativo más grande del corpus. Son pocos fragmentos, pero cada
uno pesa mucho en la reseña donde aparece.

**«Trato al personal» no es cuantificable con esta herramienta.** La regla amplia
recuperaba quejas *sobre* el personal, no críticas a *cómo se le trata*; la estrecha
deja 1 fragmento. Cualitativamente sí aparece —2 de los 60 atípicos clasificados a
mano— pero no se puede dar una cifra defendible, y se reporta solo como observación.

## 6. Limitaciones

Siete, con su alcance real.

1. **La coherencia por co-ocurrencia es prácticamente nula** (`c_npmi = −0,038`). Hay
   causa estructural —el fragmento mediano tiene 10 palabras— pero no se puede afirmar
   que los tópicos sean léxicamente coherentes. Su interpretación se sostiene en la
   lectura de fragmentos.

2. **Los temas de baja masa se miden con reglas de piso**, con falsos positivos y
   negativos documentados. Acotan el orden de magnitud; no lo miden. Dos reglas ya
   tuvieron que corregirse.

3. **La deduplicación usa un proxy débil.** Sin identificador de autor, la clave usa
   `autor_n_resenas`. Afecta a 2 filas de 2.432 y no altera resultados, pero no debe
   presentarse como exacta.

4. **Hay una fuga en la detección de idioma heredada:** un fragmento en alfabeto árabe
   pasó el filtro. Es 1 de 5.913 y quedó como atípico.

5. **El umbral de agrupamiento sesga qué atributos emergentes se ven, y el sesgo tiene
   signo:** los que forman tópico son los positivos. Interpretar los atributos
   emergentes solo por los tópicos formados subestimaría lo negativo.

6. **Cada fragmento hereda la calificación de su reseña.** El rating de un tópico no es
   la valoración del tema, sino la de las reseñas donde aparece; una reseña larga pesa
   más que una corta; y los fragmentos de una reseña no son independientes. Por eso no
   se aplica ninguna prueba de significancia.

7. **Los temas transversales no son tópicos**, y el modelado de tópicos es la
   herramienta equivocada para ellos. Su presencia se sostiene con reglas de
   recuperación y análisis cualitativo.

## Hallazgos

1. **La correspondencia entre temas emergentes y dimensiones a priori es parcial:
   AMI 0,406 sobre una línea base de cero.** Comparten estructura real, pero los tópicos
   no son una redescripción de las seis dimensiones.

2. **Ninguna dimensión se concentra en un solo tópico.** La espera es la única con
   correspondencia fuerte (62,7 % en un tópico); comida se reparte en 38 y su tópico
   principal ni siquiera es de comida. El esquema a priori y la estructura emergente
   describen el mismo corpus con cortes distintos.

3. **El 13,0 % del corpus asignado habla de atributos que el esquema no contempla**, y
   dos de ellos —medios de pago e infraestructura— son además invisibles para el
   diccionario en más del 75 % de sus fragmentos.

4. **La espera concentra el malestar** (1,88 estrellas, 75,8 % de reseñas de 1-2), y los
   atributos mejor valorados son los dos emergentes: ocasión de consumo (4,58) y
   referente en la ciudad (4,55). Ahí está la orientación más directa para la
   comunicación digital.

5. **Lo que el método no captura es desproporcionadamente crítico.** Los atípicos son
   más negativos que el corpus (3,20 contra 3,70) y los temas transversales —inocuidad
   1,62, cobro 1,87— son los peor calificados de todo el estudio. Cualquier lectura que
   se quede en los tópicos formados dará una imagen más amable de la que el corpus
   sostiene.

6. **La calidad léxica del modelo es moderada y hay que declararlo.** `c_v` 0,484 y
   `c_npmi` −0,038: la validez de los tópicos descansa en la lectura de sus fragmentos,
   no en los indicadores de coherencia.

---

*Este notebook lee de `data/processed/` los resultados ya calculados. Los scripts que
los producen están en `scripts/` y las decisiones que los sustentan, con sus cifras, en
`docs/bitacora.md` y `docs/decisiones_metodologicas.md`.*